# Entropic Regularization of Optimal Transport [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xavimaass/computational_optimal_transport_2026/blob/main/assignments/HW4.ipynb)

**Student Name:** [Your Name Here]

In this notebook, we will explore the regularized OT problem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

We consider two input histograms $a, b \in \Sigma_n$, where the probability simplex in $\mathbb{R}^n$ is defined as
$$
\Sigma_n := \left\{ a \in \mathbb{R}_+^n \;\middle|\; \sum_{i=1}^n a_i = 1 \right\}.
$$

We define the discrete entropic regularized transport cost as
$$
W_\epsilon(a, b) := \min_{P \in U(a, b)} \sum_{i,j} C_{i,j} P_{i,j} - \epsilon E(P),
$$
where the polytope of couplings is given by
$$
U(a, b) := \left\{ P \in \left(\mathbb{R}_+\right)^{n \times m} \;\middle|\; P \mathbf{1}_m = a,\; P^\top \mathbf{1}_n = b \right\},
$$
with $\mathbf{1}_n := (1, \ldots, 1)^\top \in \mathbb{R}^n$.

For $P \in \mathbb{R}_+^{n \times m}$, the entropy is defined as
$$
E(P) := -\sum_{i,j} P_{i,j} \left( \log(P_{i,j}) - 1 \right).
$$

When $\epsilon = 0$, one recovers the classical (discrete) optimal transport cost.

Remark that here we do not use the product measure as a reference but the uniform measure. By a remark discussed in class this does not change the minimizers of the problem however you may have to adapt certain formulas from the class to this different choice of reference measure.

The matrix $C \in \left(\mathbb{R}_+\right)^{n \times m}$ defines the ground cost, where $C_{i,j}$ is the cost of transporting mass from bin $i$ to bin $j$.

The regularized transport problem can be rewritten as a projection:
$$
W_\epsilon(a, b) = \epsilon \min_{P \in U(a, b)} \mathrm{KL}(P \,\|\, K),
$$
where the Gibbs kernel $K$ is defined as
$$
K_{i,j} := \exp\left( -\frac{C_{i,j}}{\epsilon} \right),
$$
and the Kullback-Leibler divergence between $P, K \in \mathbb{R}_+^{n \times m}$ is given by
$$
\mathrm{KL}(P \,\|\, K) := \sum_{i,j} P_{i,j} \left( \log\left( \frac{P_{i,j}}{K_{i,j}} \right) - 1 \right).
$$

Given a convex set $S \subset \mathbb{R}^N$, the projection of $\xi$ onto $S$ with respect to the Kullback-Leibler divergence is defined as
$$
\mathrm{Proj}_{S}^{\mathrm{KL}}(\xi) := \underset{\pi \in S}{\arg\min} \, \mathrm{KL}(\pi \,\|\, \xi).
$$

## Sinkhorn's Algorithm

A fundamental observation is that the optimality condition of the entropic regularized transport problem implies that the optimal coupling $P_\epsilon$ necessarily has the form
$$
P_\epsilon = \operatorname{diag}(u) \, K \, \operatorname{diag}(v),
$$
where the Gibbs kernel is defined as
$$
K := \exp\left( -\frac{C}{\epsilon} \right).
$$

To satisfy the marginal constraints, one must find two positive scaling vectors $u \in \mathbb{R}_+^n$ and $v \in \mathbb{R}_+^m$ such that the following equalities hold:
$$
P \mathbf{1}_m = u \odot (K v) = a, \quad \text{and} \quad
P^\top \mathbf{1}_n = v \odot (K^\top u) = b,
$$
where $\odot$ denotes element-wise (Hadamard) multiplication.

Sinkhorn's algorithm alternates between updates of these two equations and proceeds as follows:
$$
u \leftarrow \frac{a}{K v}, \quad
v \leftarrow \frac{b}{K^\top u},
$$
where the divisions are element-wise.

The algorithm is typically initialized with
$$
v = \mathbf{1}_m,
$$
where $\mathbf{1}_m$ is the vector of all ones in $\mathbb{R}^m$.  
The vector $u$ does not need to be explicitly initialized, as it is computed in the first iteration.

## Transport Between Point Clouds
We first test the method for two input measures that are uniform measures (i.e. constant histograms) supported on two point clouds (that do not necessarily have the same size).


We thus first load two points clouds $x=(x_i)_{i=1}^{n}, y=(y_i)_{i=1}^{m}, $ where $x_i, y_i \in \mathbb{R}^2$.


Number of points in each cloud, $N=(n,m)$.

In [ ]:
N = [300,200] # Number of Points
d = 2 # point cloud dimension

**Exercise 1.**

Complete the code to generate the two point clouds x and y as follows:
- x: N[0] points drawn uniformly in the square $[-0.5, 0.5]\times [-0.5, 0.5]$
- y: N[1] points drawn uniformly on an annulus of inner radius 0.8 and outer radius 1.0 (_Hint: sample $\theta \in [0, 2\pi]$ and $r \in [0.8, 1.0]$ uniformly_)

Also, complete the code for computing the Cost matrix:
$$
C_{i,j} = \|x_i - y_j\|^2,
$$
where $\|\cdot\|$ denotes the Euclidean norm in $\mathbb{R}^d$.


In [ ]:
def generate_data(N):
  """
  Generate two point clouds x and y.
  Returns:
    x: np.array of shape (2, N[0])
    y: np.array of shape (2, N[1])
  """
  # COMPLETE CODE
  x = None
  y = None
  return x, y

In [ ]:
def compute_cost_matrix(x, y):
  """
  Compute the cost matrix C.
  Returns:
  C: np.array of shape (N[0], N[1])
  """
  C = None
  return C

We now generate the dataset using the functions you completed.

In [ ]:
x, y = generate_data(N)
C = compute_cost_matrix(x, y)

We provide the following function for displaying the point clouds.

In [ ]:
def plot_point_cloud_and_transport(x, y, P = None):
  """
  Plot two point clouds and (if given) the associated transport plan.
  Arguments:
  - x: np.array of shape (2, N[0])
  - y: np.array of shape (2, N[1])
  - P: np.array of shape (N[0], N[1]) or None. Must be stochastic matrix representing transport valid plan.
  """
  plt.figure(figsize=(10,10))

  plt.scatter(x[0,:], x[1,:], s=200, edgecolors="k", c='b', linewidths=2)
  plt.scatter(y[0,:], y[1,:], s=200, edgecolors="k", c='r', linewidths=2)

  if P is not None:
    A = P * (P > np.max(P)*.8)
    i,j = np.where(A != 0)
    plt.plot([x[0,i],y[0,j]],[x[1,i],y[1,j]],'k',lw = 2)

    A = P * (P > np.max(P)*.2)
    i,j = np.where(A != 0)
    plt.plot([x[0,i],y[0,j]],[x[1,i],y[1,j]],'k:',lw = 1)

  plt.axis("off")
  plt.xlim(np.min(y[0,:])-.1,np.max(y[0,:])+.1)
  plt.ylim(np.min(y[1,:])-.1,np.max(y[1,:])+.1)

  plt.show()

In [ ]:
plot_point_cloud_and_transport(x,y)

We assign equal weight  $(a,b)$ to all points on each point cloud (i.e. uniform histograms).

In [ ]:
a = np.ones(N[0])/N[0]
b = np.ones(N[1])/N[1]

**Exercise 2.** Complete the function to compute the Gibbs Kernel

In [ ]:
def gibbs_kernel(C, epsilon):
  """
  Returns the Gibbs kernel K as defined above.
  K: np.array of shape (N[0], N[1])
  """
  # COMPLETE
  K = None
  return K

We now generate a Gibbs Kernel $K$ with regularization strength $\epsilon>0$.

In [ ]:
epsilon = .01
K = gibbs_kernel(C, epsilon)

**Exercise 3.** Implement the Sinkhorn Algorithm by completing the following function.

To monitor convergence, you need to track the evolution of the "constraint satisfaction errors":
$$
\|P \mathbf{1}_m - a\|_1 \quad \text{and} \quad \|P^\top \mathbf{1}_n - b\|_1,
$$
where  $P = \operatorname{diag}(u) K \operatorname{diag}(v)$.  Note that these residuals can be computed directly from the scaling vectors $(u, v)$ as:
$$
P \mathbf{1}_m = u \odot (K v), \quad
P^\top \mathbf{1}_n = v \odot (K^\top u).
$$

Recall that $Ax$ in numpy corresponds to `A @ x`, whereas $a \odot b$ is `a*b`

In [ ]:
def sinkhorn(a, b, K, niter=1000):
    """
    Run Sinkhorn's algorithm.
    Returns scaling vectors u (of shape (len(a),)), v (of shape (len(b),)) 
    and convergence errors Err_p (list of |P 1_m - a|_1), Err_q (list of |P^T 1_n - b|_1).
    """
    v = np.ones(len(b))
    Err_p, Err_q = [], []
    for i in tqdm(range(niter)):
        # COMPLETE THE LOOP
        u = None
    return u, v, Err_p, Err_q

In [ ]:
epsilon = 0.01
K = gibbs_kernel(C, epsilon)

u,v, Err_p, Err_q = sinkhorn(a, b, K, niter=5000)

In [ ]:
def plot_marginal_errors(err_p, err_q, eps=1e-16):
    """
    Plot log-marginal errors for P1-a and P^T1-b.
    err_p : array-like with errors for ||P 1 - a||.
    err_q : array-like with errors for ||P^T 1 - b||.
    eps (optional) : small value to avoid log(0).
    """
    err_p = np.asarray(err_p)
    err_q = np.asarray(err_q)

    fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    ax[0].plot(np.log(err_p + eps), linewidth=2)
    ax[0].set_title(r"$\|P \mathbf{1} - a\|$")
    ax[0].set_ylabel("Log Error")
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(np.log(err_q + eps), linewidth=2)
    ax[1].set_title(r"$\|P^\top \mathbf{1} - b\|$")
    ax[1].set_xlabel("Iterations")
    ax[1].set_ylabel("Log Error")
    ax[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

In [ ]:
# We plot the errors of the marginals in a log scale to better visualize convergence.
plot_marginal_errors(Err_p, Err_q)

**Exercise 4.** Implement the function to compute the optimal transport plan from u, v and K.
$P = \operatorname{diag}(u) K \operatorname{diag}(v)$
Then display it and comment on the obtained results.

In [ ]:
def transport_coupling(u, v, K):
  """Returns the optimal coupling P given by diag(u) K diag(v)"""
  # COMPLETE
  P = None
  return P

In [ ]:
# We compute it and display it.
P = transport_coupling(u, v, K)
plt.imshow(P)
plot_point_cloud_and_transport(x,y,P)

[YOUR ANSWER HERE]

## Testing multiple values of $\epsilon$.

As $\epsilon$ decreases, we expect the solution to become increasingly sparse and to concentrate around the minimal-cost entries.

**Exercise 5.** Complete the following function `compute_transport_plans` to compute the optimal transport plans for a given list of values for ɛ. Then, use the function `plot_transport_plans` to visualize them and comment on the results.

In [ ]:
def compute_transport_plans(C, a, b, epsilon_list, niter=300):
    """
    Returns a list of transport plans corresponding to different values of regularization.
    """
    plans = []
    # COMPLETE
    for eps in epsilon_list:
        P = None
        plans.append(P)

    return plans

In [ ]:
def plot_transport_plans(plans, epsilon_list, N):
    """
    Plot a list of transport plans.
    """
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    vmax = 0.3 * np.min(1 / np.asarray(N))

    for ax, P, eps in zip(axes.ravel(), plans, epsilon_list):
        # Clamp P values for visualization and display with imshow
        ax.imshow(np.clip(P, 0, vmax))
        ax.set_title(rf"$\varepsilon = {eps:.3f}$")
        ax.axis("off")

    fig.tight_layout()
    plt.show()


In [ ]:
epsilon_list = [0.1, 0.01, 0.005, 0.001]
niter = 300
plans = compute_transport_plans(C, a, b, epsilon_list, niter=niter)
plot_transport_plans(plans, epsilon_list, N)

[YOUR ANSWER HERE]

## Transport Between Histograms
We now consider a different setup, where the histogram values $a,b$ are not uniform, but the measures are defined on a uniform grid $x_i=y_i=i/n$. They are thue often refered to as "histograms".

We fix a common number of points `N=200`, and define a variable `t` which will represent the `N` points in this grid.

In [ ]:
N = 200
t = np.arange(0,N)/N

**Exercise 6.**

Define two probability histograms `a` and `b` on the grid `t` as follows:
- complete the function `gaussian_density` for it to calculate: $f(t; t_0, \sigma) =\ exp\big(\frac{-(t-t_0)^2}{2\sigma^2}\big)$
- Fix `sigma = 0.06`, and define: `a = gaussian_density(t, 0.25, sigma)` and `b = gaussian_density(t, 0.8, sigma)`.
- To ensure strict positivity, add a small `min_mass = 0.008` to `a` and `b` and then use `normalize` to turn them back into probability distributions.
Why do we need to have strictly positive mass for Sinkhorn's algorithm to be well-defined?

In [ ]:
def normalize(p):
  return p/np.sum(p)

def gaussian_density(t, t0, sigma):
  """Returns a gaussian density on the grid t
  Inputs:
  t: np.array of shape (N,)
  t0: float
  sigma: float
  Returns a np.array of shape (N,)
  """
  # COMPLETE
  return None


# COMPLETE
a,b = None, None

[YOUR ANSWER HERE]

We display the histograms you generated.

In [ ]:
def plot_histograms(t, a, b):
  plt.figure(figsize = (10,7))
  plt.subplot(2, 1, 1)
  plt.bar(t, a, width = 1/len(t), color = "darkblue")
  plt.subplot(2, 1, 2)
  plt.bar(t, b, width = 1/len(t), color = "darkblue")
  plt.show()

In [ ]:
plot_histograms(t, a, b)

We now compute the cost matrix and the Gibbs Kernel.

In [ ]:
C = compute_cost_matrix(t[np.newaxis, :],t[np.newaxis,:])
epsilon = (.03)**2
K = gibbs_kernel(C, epsilon)

**Exercise 7.** Use the Sinkhorn algorithm to find the optimal transport plan. Plot the marginal errors. Compute the transport map P. Use the function `plot_transport_map` to plot it.

In [ ]:
def plot_transport_map(P, s=None, t=None, title="Transport Plan"):
  plt.figure(figsize=(5, 5))
  plt.imshow(np.log(P + 1e-5))
  if s is not None and t is not None:
    plt.plot(s * N, t * N, 'r', linewidth=3)
  plt.title(title)
  plt.axis('off')
  plt.show()

In [ ]:
# YOUR CODE HERE

One can approximate the transport map between the two measures using the so-called **barycentric projection**. For each source point indexed by $i$, this projection is defined as:
$$
s_i := \frac{\sum_{j} P_{i,j} \, t_j}{\sum_{j} P_{i,j}}
= \frac{\left[ u \odot \left( K (v \odot t) \right) \right]_i}{a_i},
$$
where:
- $t_j \in [0,1]$ is the target value associated with index $j$,
- $\odot$ denotes element-wise (Hadamard) multiplication,
- $\frac{\cdot}{\cdot}$ denotes element-wise division.

This computation avoids explicitly constructing the full transport matrix $P$ and instead relies on efficient kernel-vector operations involving $K$, $u$, and $v$.

**Exercise 8.** Complete the following function to compute the Barycentric projection.

In [ ]:
def barycentric_projection(u, v, K, t, a):
    """
    Compute the barycentric projection as indicated above.
    Returns an array s of shape (N,) representing the barycentric projection
    """
    # YOUR CODE HERE
    s = None
    return None

In [ ]:
s = barycentric_projection(u, v, K, t, a)

We display the transport map, super-imposed over the coupling.

In [ ]:
plot_transport_map(P, s, t)

**Exercise 9.** Use the functions from the previous parts to explore the effect of varying the regularization strength $\epsilon$ on the barycentric projection and the resulting transport plan.

Try computing and visualizing the transport map for different values of $\epsilon$. 
Namely, iterate over `epsilon_values`:
1. Recompute the Gibbs Kernel `K_eps` for the specific epsilon.
2. Run Sinkhorn to get scaling vectors `u_eps` and `v_eps`.
3. Compute the coupling `P_eps` for visualization.
4. Compute the barycentric projection `s_eps`.
5. Plot the result for each value of epsilon: `plot_transport_map(P_eps, s_eps, t, title=r"Transport Map ($\varepsilon = $"+f"{epsilon_values[i]})")`

Comment on your results. You should expect that:
- Larger $\epsilon$ produces smoother, more diffused transport plans.
- Smaller $\epsilon$ results in more sharply concentrated transport, approximating the unregularized optimal transport solution.

In [ ]:
epsilon_values = [0.1, 0.01, 0.001, 0.0005]
niter = 2000

# YOUR CODE HERE

[YOUR ANSWER HERE]